In [1]:
import pandas as pd
import numpy as np
from nba_api.stats import endpoints
from great_tables import gt

In [2]:
current_year = 2026

# Create season string like "2025-26"
current_year_for_pbp = f"{current_year-1}-{str(current_year)[-2:]}"

# Read CSV
pbp_curr_yr = pd.read_csv(f"Data/{current_year_for_pbp}_pbp.csv")

# Group by gameId and fill scores within each game
pbp_curr_yr[['scoreHome', 'scoreAway']] = (
    pbp_curr_yr.groupby('gameId')[['scoreHome', 'scoreAway']]
    .ffill()
)

# Score margin
pbp_curr_yr['scoreMargin'] = pbp_curr_yr['scoreHome'] - pbp_curr_yr['scoreAway']

# Remove All-Star weekend (gameId starting with '3')
pbp_curr_yr = pbp_curr_yr[
    ~pbp_curr_yr['gameId'].astype(str).str.startswith('3')
]

# Reset index
pbp_curr_yr = pbp_curr_yr.reset_index(drop=True)

In [3]:
# --- compact_standings ---
compact_standings = (
    endpoints.leaguestandingsv3.LeagueStandingsV3(
        season=current_year_for_pbp,
        season_type="Regular Season"
    )
    .get_data_frames()[0]
    .loc[:, ["TeamID","TeamCity","TeamName","Conference","Division","WINS","LOSSES"]]
    .assign(
        TeamGP=lambda d: d.WINS + d.LOSSES,
        TeamFullName=lambda d: d.TeamCity + " " + d.TeamName
    )
)

# --- player_bio ---
player_bio = (
    endpoints.leaguedashplayerbiostats.LeagueDashPlayerBioStats(
        season=current_year_for_pbp,
        per_mode_simple="Totals"
    ).get_data_frames()[0].loc[:, "PLAYER_ID":"GP"]
)

# --- team_info ---
team_info = (
    compact_standings
    .merge(
        player_bio
        .sort_values("TEAM_ID")
        .groupby("TEAM_ID", as_index=False)
        .first()[["TEAM_ID","TEAM_ABBREVIATION"]],
        left_on="TeamID",
        right_on="TEAM_ID",
        how="left"
    )
)

# --- player_stats ---
player_stats = (
    endpoints.leaguedashplayerstats.LeagueDashPlayerStats(
        season=current_year_for_pbp,
        per_mode_detailed="Totals"
    ).get_data_frames()[0]
)

# --- player_w_gp_percentages ---
player_w_gp_percentages = (
    player_bio
    .merge(
        compact_standings.filter(regex="Team"),
        left_on="TEAM_ID",
        right_on="TeamID",
        how="left"
    )
    .merge(
        player_stats.loc[:, ["PLAYER_ID","MIN"]],
        on="PLAYER_ID",
        how="left"
    )
    .assign(G_PERCENT=lambda d: d.GP / d.TeamGP)
)

del compact_standings, player_bio, player_stats

# Awards

## The "Chicken Supplier" Award (sponsored by Los Pollos Hermanos)

most pairs of free throws missed by a visiting player in the second half (credit to livejamie for the idea)

In [4]:
missed_visitor_ft_in_second_half = (
    pbp_curr_yr
    .loc[
        (pbp_curr_yr["actionType"] == "Free Throw") &
        (pbp_curr_yr["description"].str.startswith("MISS", na=False)) &
        (pbp_curr_yr["location"] == "v") &
        (pbp_curr_yr["period"] > 2) &
        (pbp_curr_yr["subType"].str.endswith(("2", "3"), na=False))
    ]
    .groupby(["personId", "playerNameI", "gameId", "clock", "period"])
    .size()
    .reset_index(name="n")
    .query("n > 1")
    .groupby(["personId", "playerNameI"], as_index=False)
    .agg(missed_ft_pairs=("n", "size"))
    .merge(
        player_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
        .rename(columns={"PLAYER_ID": "personId","PLAYER_NAME": "player_name"}),
        on="personId",
        how="left"
    )
)

In [5]:
gt.GT(
    missed_visitor_ft_in_second_half
    .drop(columns=["playerNameI", "personId"])
    .nlargest(5, "missed_ft_pairs",keep="all")
)

missed_ft_pairs,player_name
10,Pascal Siakam
9,Rudy Gobert
8,Alperen Sengun
7,Zion Williamson
7,Ausar Thompson


## The "Rent-Free" Award (presented by Monica Geller)

team that shoots the most technical/flagrant FTAs (credit to Drummallumin for the idea)

In [6]:
# create technical + flagrant FT counts
flagrant_tech_fts = (
    pbp_curr_yr
    .loc[pbp_curr_yr["subType"].str.contains(
        "Free Throw Technical|Free Throw Flagrant", na=False
    )]
    .groupby("teamTricode", as_index=False)
    .agg(
        tech_ft=("subType", lambda s: s.str.contains("Free Throw Technical", na=False).sum()),
        flagrant_ft=("subType", lambda s: s.str.contains("Free Throw Flagrant", na=False).sum())
    )
    .assign(
        flagrant_plus_tech_fts=lambda d: d.tech_ft + d.flagrant_ft
    )
)

flagrant_tech_fts.to_csv('Output Data/Flagrant-Tech Team FTAs.csv',index=False)

In [7]:
gt.GT(flagrant_tech_fts.nlargest(5, "flagrant_plus_tech_fts",keep="all"))

teamTricode,tech_ft,flagrant_ft,flagrant_plus_tech_fts
DET,48,21,69
MEM,40,27,67
LAL,49,14,63
NYK,29,33,62
DEN,44,17,61
OKC,34,27,61
ORL,51,10,61


## The "It Ain't Over Till the Fat Lady Sings" Award (presented by Kim Kardashian)

most comeback wins where the opposing team had a lead of at least 15 points at some point in the game

In [8]:
largest_leads = (
    pbp_curr_yr
    .loc[
        pbp_curr_yr["teamTricode"].notna() &
        pbp_curr_yr["location"].isin(["h", "v"])
    ]
    .groupby("gameId")
    .agg(
        vis_final=("scoreAway", "max"),
        home_final=("scoreHome", "max"),
        largest_home_lead=("scoreMargin", "max"),
        largest_vis_lead=("scoreMargin", lambda s: -s.min()),
        home=("teamTricode", lambda s: s[pbp_curr_yr.loc[s.index, "location"] == "h"].iloc[0]),
        visitor=("teamTricode", lambda s: s[pbp_curr_yr.loc[s.index, "location"] == "v"].iloc[0]),
    )
    .reset_index()
    .assign(
        winner_team=lambda d: d["home"].where(
            d["home_final"] > d["vis_final"], d["visitor"]
        ),
        loser_team=lambda d: d["visitor"].where(
            d["home_final"] > d["vis_final"], d["home"]
        ),
        largest_winner_lead=lambda d: d["largest_home_lead"].where(
            d["home_final"] > d["vis_final"], d["largest_vis_lead"]
        ),
        largest_loser_lead=lambda d: d["largest_vis_lead"].where(
            d["home_final"] > d["vis_final"], d["largest_home_lead"]
        )
    )
)

In [9]:
gt.GT(
    largest_leads
    .groupby("winner_team", as_index=False)
    .agg(num_wins_trail_by_15=("largest_loser_lead", lambda s: (s >= 15).sum()))
    .nlargest(5, "num_wins_trail_by_15",keep="all")
)

winner_team,num_wins_trail_by_15
ORL,7
SAS,7
MIN,6
BOS,5
LAC,5
LAL,5
NYK,5


## The "Snatching Defeat from the Jaws of Victory" Award (presented by the 28-3 Atlanta Falcons)

most losses when having a lead of at least 15 points at some point in the game

In [10]:
gt.GT(
    largest_leads
    .groupby("loser_team", as_index=False)
    .agg(
        num_losses_lead_by_15=("largest_loser_lead", lambda s: (s >= 15).sum())
    )
    .nlargest(5, "num_losses_lead_by_15",keep="all")
)

loser_team,num_losses_lead_by_15
MEM,10
NOP,8
IND,7
DEN,5
LAC,5
MIA,5
MIL,5


## The Dikembe Mutombo Memorial "No Fly Zone" Award*

most blocked dunks as the blocking player

In [11]:
blocked_dunks = pbp_curr_yr.loc[
    (pbp_curr_yr["shotResult"] == "Missed") &
    (pbp_curr_yr["subType"].str.contains("Dunk", na=False))
].merge(
    pbp_curr_yr.loc[
        pbp_curr_yr["description"].str.contains("BLOCK", na=False)
    ],
    on=["gameId", "clock", "period"],
    how="inner",
    suffixes=(".x", ".y")
)

In [12]:
gt.GT(
    blocked_dunks
    .groupby(["personId.y", "playerNameI.y"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .merge(
        player_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
        .rename(columns={
            "PLAYER_ID": "personId.y",
            "PLAYER_NAME": "player_name"
        }),
        on="personId.y",
        how="left"
    )
    .drop(columns=["playerNameI.y", "personId.y"])
    .nlargest(5, "count",keep="all")
)

count,player_name
23,Brook Lopez
21,Ryan Kalkbrenner
17,Isaiah Stewart
17,Jabari Smith Jr.
12,Alex Sarr


## The Rejected for Boarding Award (sponsored by United Airlines)*

most blocked dunks as the dunking player (credit to Legdrop_soup for the idea and asw7412 for the sponsor)

In [13]:
gt.GT(
    blocked_dunks
    .groupby(["personId.x", "playerNameI.x"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .merge(
        player_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
        .rename(columns={
            "PLAYER_ID": "personId.x",
            "PLAYER_NAME": "player_name"
        }),
        on="personId.x",
        how="left"
    )
    .drop(columns=["playerNameI.x", "personId.x"])
    .nlargest(5, "count",keep="all")
)

count,player_name
10,Cody Williams
9,Rudy Gobert
9,Sidy Cissoko
9,Amen Thompson
8,OG Anunoby
8,Shaedon Sharpe
8,Moussa Diabaté


## The No Time to Taunt Award (presented by Tim Duncan)*

highest percent of blocks that stayed inbounds & recovered by blocker's team, min 0.7 blocks per game (credit to gibberisle for the idea)

In [14]:
blocks_w_next_play = (
    pbp_curr_yr
    .sort_values(["gameId","actionNumber"])
    .assign(
        next_play=lambda d: d["description"].shift(-1),
        next_location=lambda d: d["location"].shift(-1)
    )
    .loc[lambda d: d["description"].str.contains("BLOCK", na=False)]
    .assign(
        recovered_by=lambda d: (
            (~d["next_play"].str.contains("Rebound", na=False)) &
            (d["location"] == d["next_location"])
        ).map({True: "own", False: "other"})
    )
)

player_blk_info = (
    blocks_w_next_play
    .groupby(["personId", "playerNameI", "recovered_by"], as_index=False)
    .size()
    .rename(columns={"size": "blocks"})
    .pivot(
        index=["personId", "playerNameI"],
        columns="recovered_by",
        values="blocks"
    )
    .fillna(0)
)

player_blk_info.columns = [
    f"blocks_recovered_by_{c}" for c in player_blk_info.columns
]

player_blk_info = (
    player_blk_info
    .reset_index()
    .assign(
        blocks=lambda d: d.blocks_recovered_by_own + d.blocks_recovered_by_other,
        percent_blk_recovered_by_own=lambda d:
        d.blocks_recovered_by_own / d.blocks
    )
    .merge(
        player_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
        .rename(columns={"PLAYER_ID": "personId","PLAYER_NAME": "player_name"}),
        on="personId",
        how="left"
    )
    .pipe(lambda d: d.assign(player_name=d.pop("player_name")))
    .drop(columns="playerNameI")
)

average_games_played = team_info["TeamGP"].mean()

player_blk_info.to_csv('Output Data/Player Blocking Info.csv',index=False)

gt.GT(
        player_blk_info
        .loc[lambda d: d["blocks"] >= average_games_played * 0.7]
        .nlargest(5, "percent_blk_recovered_by_own",keep="all")
        .rename(columns={"blocks_recovered_by_own": "own"})
        [["player_name", "blocks", "own", "percent_blk_recovered_by_own"]]
    ).fmt_percent(columns=["percent_blk_recovered_by_own"])

player_name,blocks,own,percent_blk_recovered_by_own
Clint Capela,63.0,46.0,73.02%
Peyton Watson,61.0,39.0,63.93%
Kevin Durant,71.0,45.0,63.38%
Goga Bitadze,61.0,38.0,62.30%
Ryan Kalkbrenner,101.0,62.0,61.39%


## The Zaza Pachulia All-Ball Award*

most 3-pt shooting fouls committed (credit to watchingsongsDL, kingcobweb & An-Indian-In-The-NBA for the idea, and sunnysideoutside for the name)

In [15]:
fouls_on_threes = (
    pbp_curr_yr
    # shooting fouls
    .loc[pbp_curr_yr["description"].str.contains("S.FOUL", na=False)]
    .merge(
        pbp_curr_yr.loc[
            (
                pbp_curr_yr["description"].str.contains("Free Throw 3 of 3", na=False)
            ) |
            (
                pbp_curr_yr["description"].str.contains("3PT", na=False) &
                (pbp_curr_yr["shotResult"] == "Made")
            )
        ],
        on=["gameId", "period", "clock"],
        how="inner",
        suffixes=(".x", ".y")
    )
)

In [16]:
gt.GT(
    fouls_on_threes
    .groupby(["personId.x", "playerNameI.x"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .merge(
        player_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
        .rename(columns={
            "PLAYER_ID": "personId.x",
            "PLAYER_NAME": "player_name"
        }),
        on="personId.x",
        how="left"
    )
    .pipe(lambda d: d.assign(player_name=d.pop("player_name")))
    .drop(columns=["playerNameI.x", "personId.x"])
    .nlargest(5, "count", keep="all")
)

count,player_name
12,Peyton Watson
10,RJ Barrett
10,Quentin Grimes
10,Ryan Dunn
9,Collin Sexton
9,Jalen Suggs
9,Ja'Kobe Walter
9,Spencer Jones


## The "David vs Goliath" Award (presented by Dwyane Wade)

most shots blocked where the blocker is at least 5 inches shorter than the blockee

In [17]:
blocked_shots_w_player_heights = (
    pbp_curr_yr
    # missed shots
    .loc[pbp_curr_yr["shotResult"] == "Missed"]
    # join with block events
    .merge(
        pbp_curr_yr.loc[pbp_curr_yr["description"].str.contains("BLOCK", na=False)],
        on=["gameId", "clock", "period"],
        how="inner",
        suffixes=(".x", ".y")
    )
    # height of the shooter (blocked player)
    .merge(
        player_w_gp_percentages[["PLAYER_ID", "PLAYER_HEIGHT_INCHES"]]
        .rename(columns={
            "PLAYER_ID": "personId.x",
            "PLAYER_HEIGHT_INCHES": "blocked_player_height"
        }),
        on="personId.x",
        how="left"
    )
    # height of the blocker
    .merge(
        player_w_gp_percentages[["PLAYER_ID", "PLAYER_HEIGHT_INCHES"]]
        .rename(columns={
            "PLAYER_ID": "personId.y",
            "PLAYER_HEIGHT_INCHES": "blocking_player_height"
        }),
        on="personId.y",
        how="left"
    )
    .assign(
        height_diff=lambda d:
        d.blocked_player_height - d.blocking_player_height
    )
)

In [18]:
gt.GT(
        blocked_shots_w_player_heights
        .loc[lambda d: d["height_diff"] >= 5]
        .groupby(["personId.y", "playerNameI.y"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .merge(
            player_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
            .rename(columns={
                "PLAYER_ID": "personId.y",
                "PLAYER_NAME": "player_name"
            }),
            on="personId.y",
            how="left"
        )
        .pipe(lambda d: d.assign(player_name=d.pop("player_name")))
        .drop(columns=["playerNameI.y", "personId.y"])
        .nlargest(5, "count", keep="all")
    )

count,player_name
26,Derrick White
14,Tyrese Maxey
14,Keon Ellis
13,Anthony Edwards
13,Craig Porter Jr.
13,Reed Sheppard


## The "Call Game" Award (presented by Paul "No OT Tonight" George)

most game winning points (defined as the first points that eclipsed the losing team's total) (credit to Necessary_Career_253 for the idea & Clownp3nis for the presenter)

In [19]:
game_winning_shots = (
    pbp_curr_yr
    .assign(
        vis_final=lambda d: d.groupby("gameId")["scoreAway"].transform("max"),
        home_final=lambda d: d.groupby("gameId")["scoreHome"].transform("max")
    )
    .assign(
        winner=lambda d: d["home_final"].gt(d["vis_final"]).map(
            {True: "home", False: "visitor"}
        ),
        loser_points=lambda d: d["vis_final"].where(
            d["home_final"] > d["vis_final"], d["home_final"]
        )
    )
    .loc[
        lambda d:
        ((d["winner"] == "home") & (d["scoreHome"] > d["loser_points"])) |
        ((d["winner"] == "visitor") & (d["scoreAway"] > d["loser_points"]))
    ]
    .sort_values("actionId")
    .groupby("gameId", as_index=False)
    .first()
    .assign(
        shotValue=lambda d: d["shotValue"].where(
            d["actionType"] != "Free Throw", 1
        )
    )
)

game_win_shot_summary = (
    game_winning_shots
    .groupby(["personId", "playerNameI"], as_index=False)
    .agg(
        game_winning_3=("shotValue", lambda s: (s == 3).sum()),
        game_winning_2=("shotValue", lambda s: (s == 2).sum()),
        game_winning_ft=("shotValue", lambda s: (s == 1).sum()),
        game_winning_points=("shotValue", "sum")
    )
    .merge(
        player_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
        .rename(columns={
            "PLAYER_ID": "personId",
            "PLAYER_NAME": "player_name"
        }),
        on="personId",
        how="left"
    )
    .pipe(lambda d: d.assign(player_name=d.pop("player_name")))
    .drop(columns="playerNameI")
)

game_win_shot_summary.to_csv('Output Data/Game-Winning Shots.csv',index=False)

In [20]:
gt.GT(
        game_win_shot_summary
        .nlargest(5, "game_winning_points", keep="all")
        .drop(columns="personId")
    ).cols_label(
        game_winning_3="GW3",
        game_winning_2="GW2",
        game_winning_ft="GWFT",
        game_winning_points="GWPTS"
    )

GW3,GW2,GWFT,GWPTS,player_name
7,7,4,39,Anthony Edwards
7,5,6,37,Shai Gilgeous-Alexander
4,8,5,33,Donovan Mitchell
4,6,6,30,Desmond Bane
4,7,1,27,Jalen Johnson


## The Bowling Ball Award (sponsored by Pete Weber, presented by Glen "Big Baby" Davis)*

most charges committed (credit to Kdog122025 for the idea)

In [21]:
violations = (
    pbp_curr_yr
    .loc[
        pbp_curr_yr["description"].str.contains(
            "Charge|Goaltending|Kick|Traveling|3 Second",
            na=False
        )
    ]
    .merge(
        player_w_gp_percentages[["PLAYER_ID", "PLAYER_NAME"]]
        .rename(columns={
            "PLAYER_ID": "personId",
            "PLAYER_NAME": "player_name"
        }),
        on="personId",
        how="left"
    )
    .pipe(lambda d: d.assign(player_name=d.pop("player_name")))
    .drop(columns="playerNameI")
)

In [22]:
gt.GT(
    violations
    .loc[lambda d: d["description"].str.contains("Charge", na=False)]
    .groupby("player_name", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .nlargest(5, "count", keep="all")
)

player_name,count
Karl-Anthony Towns,15
Derik Queen,13
Victor Wembanyama,11
Zion Williamson,10
Alperen Sengun,9
Deni Avdija,9
Jaren Jackson Jr.,9
Jaylen Brown,9
Pascal Siakam,9


## "The Good Ol' Hockey Game, is the Best Game You Can Name" Award (presented by Dominik Hasek)*

most goaltends committed (credit to Kdog122025 for the idea)

In [23]:
gt.GT(
    violations
    .loc[lambda d: d["description"].str.contains("Goaltending", na=False)]
    .groupby("player_name", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .nlargest(5, "count", keep="all")
)

player_name,count
Kel'el Ware,23
Alex Sarr,16
Moussa Diabaté,16
Nic Claxton,15
Mark Williams,14


## "The Thing about Arsenal Is They Always Try to Walk It In" Award (presented by MLS Commissioner Don Garber)*

most kicked ball violations

In [24]:
gt.GT(
    violations
    .loc[lambda d: d["description"].str.contains("Kick", na=False)]
    .groupby("player_name", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .nlargest(5, "count", keep="all")
)

player_name,count
Bam Adebayo,17
Victor Wembanyama,16
Alperen Sengun,12
Nikola Vučević,12
Mark Williams,11


## The "Pack Your Bags" Award (Sponsored by Emirates, the global airline partner of the NBA)

Player with the most traveling calls

In [25]:
gt.GT(
    violations
    .loc[lambda d: d["description"].str.contains("Traveling", na=False)]
    .groupby("player_name", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .nlargest(5, "count", keep="all")
)

player_name,count
Jalen Johnson,21
Jaylen Brown,16
Kyle Kuzma,14
Coby White,13
Keyonte George,13
Michael Porter Jr.,13


## The "Holy Hand Grenade of Antioch" Award (Sponsored by the National Parks System, presented by Sesame Street's Count von Count)

Player with the most illegal defense/defensive 3-second calls (credit to PsychoM & MrBuckBuck for the idea, poinsetia)

In [26]:
gt.GT(
    violations
    .loc[lambda d: d["description"].str.contains("3 Second", na=False)]
    .groupby("player_name", as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .nlargest(5, "count", keep="all")
)

player_name,count
Deandre Ayton,7
Karl-Anthony Towns,6
Jalen Duren,4
Maxime Raynaud,4
Moussa Diabaté,4
Myles Turner,4
Nic Claxton,4
Rudy Gobert,4


## The "Time is An Illusion" Award (sponsored by Salvador Dali)

team with the most 24-second shotclock, 8-second backcourt and 5-second inbound violations (credit to Necessary_Career_253 for the idea)

In [27]:
team_violations = (
    pbp_curr_yr
    .loc[
        (pbp_curr_yr["actionType"] == "Turnover") &
        (pbp_curr_yr["subType"].str.contains("Shot Clock|5 Second|8 Second", na=False))
    ]
    .assign(
        team_id=lambda d: (
            np.where(d['teamId'] == 0,d['personId'],d['teamId'])
        )
    )
)

team_violations_summary=(team_violations
    .groupby("team_id", as_index=False)
    .agg(
        shot_clock=("subType", lambda s: s.str.contains("Shot Clock", na=False).sum()),
        inbound=("subType", lambda s: s.str.contains("5 Second", na=False).sum()),
        halfcourt=("subType", lambda s: s.str.contains("8 Second", na=False).sum())
    )
    .assign(
        tot_violations=lambda d:
        d.shot_clock + d.inbound + d.halfcourt
    ).merge(
        right=team_info[['TeamID','TeamFullName']],how='left',left_on='team_id',right_on='TeamID'
    )
    [['TeamFullName','shot_clock','inbound','halfcourt','tot_violations']]
)

team_violations_summary.to_csv('Output Data/Team Violations Summary.csv',index=False)

In [28]:
gt.GT(
    team_violations_summary
    .nlargest(5, "tot_violations", keep="all")
)

TeamFullName,shot_clock,inbound,halfcourt,tot_violations
Houston Rockets,72,4,5,81
Charlotte Hornets,77,2,1,80
Detroit Pistons,75,2,2,79
Phoenix Suns,77,0,1,78
Boston Celtics,64,5,2,71


## The "I'll Have It to Go" Award (sponsored by DoorDash)

coach with lowest timeout utilization (credit to xfinityhomeboy, Ill_Ad3517 & s-sea (who also came up with the name))

In [29]:
num_timeouts_available = (
    pbp_curr_yr
    .loc[pbp_curr_yr["teamTricode"].notna()]
    .sort_values(["gameId", "period"])
    .groupby(["gameId", "period", "teamTricode"], as_index=False)
    .first()[["gameId", "period", "teamTricode"]]
    .rename(columns={"teamTricode": "team"})
    .groupby("team", as_index=False)
    .agg(
        games=("period", lambda s: (s <= 4).sum() / 4),
        overtimes=("period", lambda s: (s > 4).sum())
    )
    .assign(num_timeouts=lambda d: 7 * d.games + 2 * d.overtimes)
    .merge(
        team_info[["TeamName", "TEAM_ABBREVIATION"]],
        left_on="team",
        right_on="TEAM_ABBREVIATION",
        how="left"
    )
)

num_timeouts_used = (
    pbp_curr_yr
    .loc[
        pbp_curr_yr["description"].str.contains("Timeout", na=False) &
        ~pbp_curr_yr["description"].str.contains("Excess", na=False)
    ]
    [["gameId", "period", "description"]]
    .assign(
        team_name=lambda d: (
            d["description"]
            .str.split(" Timeout:", n=1)
            .str[0]
            .str.title()
        )
    )
    .groupby("team_name", as_index=False)
    .size()
    .rename(columns={"size": "num_timeouts_used"})
)

In [30]:
timeouts_used_percent = (
    num_timeouts_available
    .merge(
        num_timeouts_used,
        left_on="TeamName",
        right_on="team_name",
        how="left"
    )
    .assign(
        timeout_use_percent=lambda d:
        d["num_timeouts_used"] / d["num_timeouts"]
    )
    .sort_values("timeout_use_percent", ascending=False)
)

gt.GT(
    timeouts_used_percent
    [["team", "num_timeouts", "num_timeouts_used", "timeout_use_percent"]]
    .nsmallest(5, "timeout_use_percent", keep="all")
).fmt_percent(columns="timeout_use_percent")

team,num_timeouts,num_timeouts_used,timeout_use_percent
UTA,586.0,399.0,68.09%
DET,584.0,404.0,69.18%
HOU,592.0,416.0,70.27%
BOS,578.0,410.0,70.93%
NYK,587.0,419.0,71.38%


## The "Hotheaded" Award (presented by Don Nelson)

most technicals plus ejections for non-players (credit to livejamie for the idea)

In [31]:
coach_ejection_techs = (
    pbp_curr_yr
    .loc[
        (pbp_curr_yr["teamId"] == 0) &
        (
            (pbp_curr_yr["actionType"] == "Ejection") |
            pbp_curr_yr["description"].str.contains(
                "T.FOUL|DOUBLE.TECHNICAL.FOUL",
                na=False
            )
        )
    ]
    .assign(
        coach=lambda d: (
            d["description"]
            .where(
                d["actionType"] == "Ejection",
                d["description"].str.split(" Foul:", n=1).str[0]
            )
            .where(
                d["actionType"] != "Ejection",
                d["description"].str.split(" Ejection:", n=1).str[0]
            )
            .str.title()
        )
    )
)

coach_ejection_tech_summary=(
    coach_ejection_techs
    .groupby("coach", as_index=False)
    .agg(
        ejections=("actionType", lambda s: (s == "Ejection").sum()),
        technicals=("actionType", lambda s: (s == "Foul").sum())
    )
    .assign(techs_plus_eject=lambda d: d.technicals + d.ejections)
)

coach_ejection_tech_summary.to_csv('Output Data/Coach Ejection Tech Summary.csv',index=False)

In [32]:
gt.GT(coach_ejection_tech_summary.nlargest(5, "techs_plus_eject", keep="all"))

coach,ejections,technicals,techs_plus_eject
Ime Udoka,2,12,14
John-Blair Bickerstaff,0,11,11
Kenny Atkinson,2,9,11
Chris Finch,1,9,10
Rick Carlisle,1,8,9
